# Prepare multiresolution network traffic

Start with the CESNET ten-minute series in `c10m_ideal.csv` and construct
hourly, six-hourly, and daily sums. Each input row needs a `date` and a
`total_traffic_kbps` value.

The input is the selected, prepared series. Preparing it from raw CESNET
archives also requires the institution, subnet, or IP selection and the
field conversion used for that series; see
[Dataset preparation](../docs/DATASETS.md).

Run the cells in order from the repository root or `notebooks/` directory.

In [ ]:
from pathlib import Path
import sys

import pandas as pd

ROOT = next(
    (path for path in (Path.cwd(), Path.cwd().parent)
     if (path / "tools" / "prepare_datasets.py").is_file()),
    None,
)
if ROOT is None:
    raise RuntimeError("Open this notebook from the repository root or notebooks directory.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from tools.prepare_datasets import write_outputs

## Input and output

Place `c10m_ideal.csv` in `data/Network/`, or update `INPUT_PATH` below.
Set `OVERWRITE = True` to replace existing prepared outputs.

In [ ]:
INPUT_PATH = ROOT / "data" / "Network" / "c10m_ideal.csv"
OUTPUT_DIR = ROOT / "data" / "Network"
OVERWRITE = False

## Load and aggregate

Sort the timestamps and retain their timezone. The hierarchy uses factors
`[1, 6, 36, 144]` for ten minutes, one hour, six hours, and one day.

In [ ]:
raw = pd.read_csv(INPUT_PATH, parse_dates=["date"])
if (INPUT_PATH.resolve() == (OUTPUT_DIR / "c10m_ideal.csv").resolve()
        and not raw["date"].is_monotonic_increasing):
    raise ValueError(
        "The finest-resolution input must be in chronological order. "
        "Use a separate input path to save a sorted copy in OUTPUT_DIR."
    )
finest = raw[["date", "total_traffic_kbps"]].sort_values("date").set_index("date")
if finest.index.has_duplicates:
    raise ValueError("Input timestamps must be unique.")

outputs = {"c10m_ideal.csv": finest}
for resolution, rule in (("1h", "1h"), ("6h", "6h"), ("1d", "1D")):
    outputs[f"c{resolution}_ideal.csv"] = finest.resample(
        rule, origin="start_day", label="left", closed="left"
    ).sum()

## Save the CSVs

Keep the finest-resolution input when it already occupies its output path.
Write the other files to `OUTPUT_DIR`.

In [ ]:
to_write = {
    name: frame for name, frame in outputs.items()
    if (OUTPUT_DIR / name).resolve() != INPUT_PATH.resolve()
}
write_outputs(to_write, OUTPUT_DIR, overwrite=OVERWRITE)

In [ ]:
summary = pd.DataFrame([
    {"file": name, "rows": len(frame),
     "start": frame.index[0], "end": frame.index[-1]}
    for name, frame in outputs.items()
])
summary

## Training split and normalization

The study's CESNET test period starts at `2024-04-07 00:00:00+0000`.
The loader uses the final 10% of the preceding observations for validation
and fits normalization on the remaining training observations. It constructs
LACE's coarse targets by summing the transformed finest-resolution future
windows. Keep the prepared CSVs in physical units.